# Harbour Surveillance - Standalone T2-T7 Tracking Solution

This notebook uses the scenario JSON files generated by the provided school notebook in `harbour_sim_output/`. The original `harbour_simulation.ipynb` is not modified.

Implemented here:
- T2 coordinate frame manager tests.
- T3 radar-only EKF on Scenario A.
- T4 radar + stereo camera fusion on Scenario B, sequential and joint.
- T5 radar + camera + AIS asynchronous fusion on Scenario C, including AIS dropout.
- T6 gating and data association on Scenario D.
- T7 track lifecycle and MOTP/CE validation on Scenario E.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

ROOT = Path.cwd()
if ROOT.name == "tracking_solution":
    ROOT = ROOT.parent

TRACKING_DIR = ROOT / "tracking_solution"
SCENARIO_DIR = ROOT / "harbour_sim_output"
sys.path.insert(0, str(TRACKING_DIR))

from harbour_tracking import (
    ScenarioData,
    first_accepted_after,
    max_history_gap,
    multi_target_metrics,
    multi_target_summary,
    rmse,
    run_multi_target_tracking,
    run_tracking,
)
from run_t2_t5 import run_t2_unit_tests, result_summary

print(f"Repository root: {ROOT}")
print(f"Scenario data:   {SCENARIO_DIR}")

## 1. Load School Scenario Files

In [ ]:
scenario_a = ScenarioData.load(SCENARIO_DIR / "scenario_A.json")
scenario_b = ScenarioData.load(SCENARIO_DIR / "scenario_B.json")
scenario_c = ScenarioData.load(SCENARIO_DIR / "scenario_C.json")
scenario_d = ScenarioData.load(SCENARIO_DIR / "scenario_D.json")
scenario_e = ScenarioData.load(SCENARIO_DIR / "scenario_E.json")

for scenario in (scenario_a, scenario_b, scenario_c, scenario_d, scenario_e):
    counts = {}
    for measurement in scenario.measurements:
        counts[measurement.sensor_id] = counts.get(measurement.sensor_id, 0) + 1
    print(
        f"Scenario {scenario.scenario_name}: "
        f"t_end={scenario.t_end:.1f}s, "
        f"targets={len(scenario.ground_truth)}, "
        f"measurements={counts}"
    )

## 2. T2 - Coordinate Frame Manager

In [ ]:
t2_results = run_t2_unit_tests()
t2_results

The AIS update is handled as the assignment requests: AIS NED reports are converted to implied range/bearing relative to the nearest vessel GNSS fix. Because the measurement units change, the AIS Cartesian covariance is transformed through the polar Jacobian before the EKF update.

## 3. T3 - Scenario A, Radar-Only EKF

In [ ]:
t3 = run_tracking(
    scenario_a,
    allowed_sensors=("radar",),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T3 radar only",
)

t3_summary = result_summary(t3, scenario_a, start_time=20.0)
t3_confirm_limit = t3.bootstrap_time + 5.0 * (1.0 / 0.3)
t3_pass = {
    "confirmation": t3.confirmation_time is not None and t3.confirmation_time <= t3_confirm_limit,
    "rmse": t3_summary["rmse_after_start_m"] < 12.0,
    "nis": t3_summary["nis_inside_95_pct"] >= 90.0,
}

print(json.dumps(t3_summary, indent=2))
print("pass:", t3_pass)

Track initiation is measurement-only. It does not use `target_id` or `is_false_alarm` labels from the simulation JSON.

## 4. T4 - Scenario B, Radar + Stereo Camera

In [ ]:
t4_radar = run_tracking(
    scenario_b,
    allowed_sensors=("radar",),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T4 radar baseline",
)
t4_seq = run_tracking(
    scenario_b,
    allowed_sensors=("radar", "camera"),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T4 sequential",
)
t4_joint = run_tracking(
    scenario_b,
    allowed_sensors=("radar", "camera"),
    fusion_mode="joint",
    bootstrap_sensors=("radar",),
    name="T4 joint",
)

t4_summary = {
    "radar_only": result_summary(t4_radar, scenario_b, start_time=5.0),
    "sequential": result_summary(t4_seq, scenario_b, start_time=5.0),
    "joint": result_summary(t4_joint, scenario_b, start_time=5.0),
    "rmse_window_0_20_m": {
        "radar_only": round(rmse(t4_radar, scenario_b, start_time=0.0, end_time=20.0), 3),
        "sequential": round(rmse(t4_seq, scenario_b, start_time=0.0, end_time=20.0), 3),
        "joint": round(rmse(t4_joint, scenario_b, start_time=0.0, end_time=20.0), 3),
    },
}

print(json.dumps(t4_summary, indent=2))

The camera only has true detections during the early visible part of Scenario B. The local RMSE window `0-20 s` therefore shows the camera benefit most clearly.

## 5. T5 - Scenario C, Add AIS and Handle Dropout

In [ ]:
t5_no_ais = run_tracking(
    scenario_c,
    allowed_sensors=("radar", "camera"),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T5 no AIS",
)
t5_with_ais = run_tracking(
    scenario_c,
    allowed_sensors=("radar", "camera", "ais"),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T5 with AIS",
)
t5_ais_only = run_tracking(
    scenario_c,
    allowed_sensors=("ais",),
    fusion_mode="sequential",
    bootstrap_sensors=("ais",),
    name="T5 AIS-only initiation",
)

t5_summary = {
    "without_ais": result_summary(t5_no_ais, scenario_c, start_time=20.0),
    "with_ais": result_summary(t5_with_ais, scenario_c, start_time=20.0),
    "ais_only_initiation": result_summary(t5_ais_only, scenario_c, start_time=20.0),
    "rmse_available_windows_m": {
        "without_ais": round(
            np.mean([
                rmse(t5_no_ais, scenario_c, start_time=20.0, end_time=60.0),
                rmse(t5_no_ais, scenario_c, start_time=90.0),
            ]),
            3,
        ),
        "with_ais": round(
            np.mean([
                rmse(t5_with_ais, scenario_c, start_time=20.0, end_time=60.0),
                rmse(t5_with_ais, scenario_c, start_time=90.0),
            ]),
            3,
        ),
    },
    "dropout_60_90": {
        "rmse_without_ais_m": round(rmse(t5_no_ais, scenario_c, start_time=60.0, end_time=90.0), 3),
        "rmse_with_ais_m": round(rmse(t5_with_ais, scenario_c, start_time=60.0, end_time=90.0), 3),
        "max_history_gap_s": round(max_history_gap(t5_with_ais, 60.0, 90.0), 3),
        "accepted_radar_camera_updates": sum(
            1
            for row in t5_with_ais.history
            if 60.0 <= row.time <= 90.0 and ("radar" in row.sensors or "camera" in row.sensors)
        ),
    },
    "first_ais_reacquisition_after_90_s": first_accepted_after(t5_with_ais, "ais", 90.0),
}

print(json.dumps(t5_summary, indent=2))

The AIS dropout is `60-90 s`. The tracker continues on radar/camera during that interval and accepts AIS again at the first AIS scan after the blackout.

## 6. T6 - Scenario D, Gating and Data Association

In [ ]:
lifecycle = {
    "confirmation_m": 4,
    "confirmation_n": 6,
    "tentative_delete_after": 2,
    "confirmed_delete_after": 5,
}

t6_d = run_multi_target_tracking(
    scenario_d,
    allowed_sensors=("radar", "camera"),
    name="T6/T7 Scenario D",
    **lifecycle,
)
metrics_d = multi_target_metrics(t6_d, scenario_d, start_time=0.0)
summary_d = multi_target_summary(t6_d, scenario_d, start_time=0.0)
pass_d = {
    "motp": summary_d["avg_motp_m"] < 15.0,
    "ce": summary_d["avg_ce"] < 0.5,
    "identity": summary_d["id_switches"] == 0,
    "final_cardinality": summary_d["final_confirmed_tracks"] == 4,
}

print(json.dumps(summary_d, indent=2))
print("pass:", pass_d)

The data association stage uses Mahalanobis gating at `P_G = 0.99` and a Global Nearest Neighbour assignment for each active sensor stream. The lifecycle is configurable; this high-clutter run uses `4-of-6` confirmation to avoid confirming random clutter tracks.

## 7. T7 - Scenario E, Full Track Lifecycle

In [ ]:
t7_e = run_multi_target_tracking(
    scenario_e,
    allowed_sensors=("radar", "camera", "ais"),
    name="T6/T7 Scenario E",
    **lifecycle,
)
metrics_e = multi_target_metrics(t7_e, scenario_e, start_time=0.0)
summary_e = multi_target_summary(t7_e, scenario_e, start_time=0.0)
pass_e = {
    "motp": summary_e["avg_motp_m"] < 20.0,
    "ce": summary_e["avg_ce"] < 1.0,
    "identity": summary_e["id_switches"] == 0,
    "final_cardinality": summary_e["final_confirmed_tracks"] == 5,
}

print(json.dumps(summary_e, indent=2))
print("pass:", pass_e)

At the end of Scenario E, target 4 has left the scene, so five confirmed tracks at `t=180 s` is the correct final cardinality.

## 8. Save Report

In [ ]:
report = {
    "T2": t2_results,
    "T3": t3_summary,
    "T3_pass": t3_pass,
    "T4": t4_summary,
    "T5": t5_summary,
    "T6_scenario_D": {
        "summary": summary_d,
        "pass": pass_d,
        "motp_series": metrics_d["motp_series"],
        "ce_series": metrics_d["ce_series"],
    },
    "T7_scenario_E": {
        "summary": summary_e,
        "pass": pass_e,
        "motp_series": metrics_e["motp_series"],
        "ce_series": metrics_e["ce_series"],
    },
}

out_path = TRACKING_DIR / "results_t2_t7.json"
out_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Wrote {out_path}")